## Import thư viện

In [17]:
# Cell 1 - Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

## Đọc dữ liệu

In [18]:
# Cell 2 - Load dataset
INPUT_FILE = "../news_dataset.json" 

if INPUT_FILE.endswith(".json"):
    df = pd.read_json(INPUT_FILE)
elif INPUT_FILE.endswith(".csv"):
    df = pd.read_csv(INPUT_FILE)
else:
    raise ValueError("Chỉ hỗ trợ .json hoặc .csv")

print(df.shape)
print(df.columns.tolist())
df.head(3)

(184539, 10)
['id', 'author', 'content', 'picture_count', 'processed', 'source', 'title', 'topic', 'url', 'crawled_at']


,id,author,content,picture_count,processed,source,title,topic,url,crawled_at
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,2022-08-01 09:09:22.817308
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,2022-08-01 09:09:21.181469
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,2022-08-01 09:09:15.311901


## Chuẩn hóa tên cột

In [19]:
# Cell 3 - Normalize column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

rename_map = {
    "id": "id",
    "author": "author",
    "title": "title",
    "content": "content",
    "picture_count": "picture_count",
    "topic": "topic",
    "url": "url",
    "source": "source",
    "crawled_at": "crawled_at",
    "processed": "processed"
}
df = df.rename(columns=rename_map)

print(df.columns.tolist())

['id', 'author', 'content', 'picture_count', 'processed', 'source', 'title', 'topic', 'url', 'crawled_at']


## Kiểm tra topic hiện có

In [20]:
# Cell 4 - Inspect topic values
topic_stats = (
    df["topic"]
    .fillna("missing")
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
    .reset_index()
)
topic_stats.columns = ["topic", "count"]
topic_stats["ratio"] = (topic_stats["count"] / len(df)).round(4)

topic_stats.head(50)

,topic,count,ratio
0,,39488,0.2140
1,Thế giới,10722,0.0581
2,Thể thao,10171,0.0551
3,Xã hội,7708,0.0418
4,Pháp luật,7442,0.0403
5,Thời sự,7204,0.0390
6,Kinh doanh,5793,0.0314
7,Giải trí,5765,0.0312
8,Sức khỏe,4929,0.0267
9,None,4231,0.0229


## Khai báo rule gán nhãn
Với dataset này, "Bất động sản" là chủ đề biên khá mơ hồ, nhưng với lab này thì mình chọn không dùng label này

In [21]:
# Cell 5 - Define topic groups for binary labeling

positive_topics = {
    "kinh doanh",
    "kinh tế",
    "tài chính - kinh doanh",
    "khanh doanh", 
    "kinh doanh ",
    "kinh tế ",
    "kinh doanh quốc tế",
    "thị trường",
    "chứng khoán",
    "tài chính",
    "doanh nghiệp"
}

## Chuẩn hóa topic và gán nhãn

In [22]:
df["topic_clean"] = (
    df["topic"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Các giá trị coi như thiếu topic
missing_topic_values = {"", "none", "null", "nan"}

df.loc[df["topic_clean"].isin(missing_topic_values), "topic_clean"] = np.nan

df["label"] = np.where(
    df["topic_clean"].isin(positive_topics), 1,
    np.where(df["topic_clean"].isna(), np.nan, 0)
)

df[["topic", "topic_clean", "label"]].sample(10, random_state=42)

,topic,topic_clean,label
173457,None,NaN,NaN
65544,Sức khỏe,sức khỏe,0.0
119515,,NaN,NaN
35692,Quân sự,quân sự,0.0
158627,Nhịp sống,nhịp sống,0.0
150400,Thể thao,thể thao,0.0
138389,Quốc tế,quốc tế,0.0
153570,Công nghệ,công nghệ,0.0
73202,Kinh doanh,kinh doanh,1.0
147985,XÃ HỘI,xã hội,0.0


## Kiểm tra phân bố nhãn sau khi map

In [23]:
label_stats = (
    df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

label_stats["ratio"] = (label_stats["count"] / len(df)).round(4)
label_stats

,label,count,ratio
0,0.0,128697,0.6974
1,NaN,43719,0.2369
2,1.0,12123,0.0657


## Kiểm tra các topic được gán label = 1

In [24]:
positive_topic_stats = (
    df[df["label"] == 1]["topic_clean"]
    .value_counts()
    .reset_index()
)

positive_topic_stats.columns = ["topic_clean", "count"]
positive_topic_stats["ratio"] = (positive_topic_stats["count"] / len(df)).round(4)
positive_topic_stats

,topic_clean,count,ratio
0,kinh doanh,6451,0.0350
1,kinh tế,4208,0.0228
2,tài chính - kinh doanh,934,0.0051
3,doanh nghiệp,275,0.0015
4,tài chính,151,0.0008
5,thị trường,57,0.0003
6,chứng khoán,47,0.0003


## Kiểm tra các topic thiếu / chưa rõ

In [25]:
unknown_topic_df = df[df["topic_clean"].isna()].copy()

print("Số dòng thiếu topic:", len(unknown_topic_df))
unknown_topic_df[["id", "title", "topic", "source", "url"]].head(10)

Số dòng thiếu topic: 43719


,id,title,topic,source,url
8,218262,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,,thethaovanhoa,https://thethaovanhoa.vn/du-doan-bong-da/soi-k...
10,218260,Hà Nội: Người lao động than trời vì khu nhà ở ...,,danviet,https://danviet.vn/ha-noi-nguoi-lao-dong-than-...
12,218258,Ukraine mất khách hàng lớn mua ngũ cốc,,soha,https://soha.vn/ukraine-mat-khach-hang-lon-mua...
16,218254,Chuông vàng vọng cổ năm 2022: Có gì để chờ đợi?,,thethaovanhoa,https://thethaovanhoa.vn/van-hoa/chuong-vang-v...
19,218251,Xác định danh tính người phụ nữ tử vong trên s...,,zingnews,https://zingnews.vn/xac-dinh-danh-tinh-nguoi-p...
24,218246,Bóng đá Anh giành chức vô địch EURO đầu tiên t...,,bongdaplus,https://bongdaplus.vn/euro-cup-chau-au/bong-da...
25,218245,Sáng Phương Nam - 01/8/2022 - Video đã phát tr...,,vtv.vn,https://vtv.vn/video/sang-phuong-nam-01-8-2022...
30,218240,Tài xế gặp sự cố khi qua cao tốc Long Thành - ...,,zingnews,https://zingnews.vn/video-tai-xe-gap-su-co-khi...
51,218219,Yêu cầu rà soát loạt dự án chắn biển Nha Trang,,vtv.vn,https://vtv.vn/xa-hoi/yeu-cau-ra-soat-loat-du-...
52,218218,Đằng sau chuyện khát lao động diện rộng nhiều ...,,dantri,https://dantri.com.vn/kinh-doanh/dang-sau-chuy...


## Tạo cột desc và text_input
Dataset này không có desc riêng, nên tạm lấy đoạn đầu của content

In [26]:
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["content"] = df["content"].fillna("").astype(str).str.strip()

df["desc"] = df["content"].str.slice(0, 300)

## Tạo dataframe đúng format bài tập

In [27]:
final_df = df[[
    "id",
    "url",
    "topic_clean",
    "label",
    "title",
    "desc",
    "content",
    "source",
    "author",
    "crawled_at"
]].copy()

final_df = final_df.rename(columns={
    "topic_clean": "category",
    "content": "text"
})

final_df.head(5)

,id,url,category,label,title,desc,text,source,author,crawled_at
0,218270,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,pháp luật,0.0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...","Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...","Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",docbao.vn,,2022-08-01 09:09:22.817308
1,218269,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,sống kết nối,0.0,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G","Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...","Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",vtc.vn,(Nguồn: Sina),2022-08-01 09:09:21.181469
2,218268,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,giáo dục,0.0,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,thanhnien.vn,Hồ Sỹ Anh,2022-08-01 09:09:15.311901
3,218267,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,thế giới,0.0,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,vnexpress,Ngọc Ánh,2022-08-01 09:09:02.211498
4,218266,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,thời sự - xã hội,0.0,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,soha,HẢI YẾN - MINH LÝ,2022-08-01 09:09:01.601170


## Loại bỏ dòng chưa có nhãn

In [28]:
labeled_df = final_df.dropna(subset=["label"]).copy()
labeled_df["label"] = labeled_df["label"].astype(int)

print("Tổng số mẫu sau khi bỏ topic thiếu:", len(labeled_df))
labeled_df["label"].value_counts().sort_index()

Tổng số mẫu sau khi bỏ topic thiếu: 140820


label
0    128697
1     12123
Name: count, dtype: int64

## Kiểm tra chất lượng dữ liệu

In [29]:
quality_report = {
    "total_rows": len(labeled_df),
    "duplicate_id": labeled_df["id"].duplicated().sum(),
    "duplicate_url": labeled_df["url"].duplicated().sum(),
    "empty_title": (labeled_df["title"].str.len() == 0).sum(),
    "empty_text": (labeled_df["text"].str.len() == 0).sum(),
    "short_text_<200_chars": (labeled_df["text"].str.len() < 200).sum(),
}

pd.DataFrame([quality_report])

,total_rows,duplicate_id,duplicate_url,empty_title,empty_text,short_text_<200_chars
0,140820,0,0,1,9565,10735


## Làm sạch cơ bản

In [30]:
clean_df = labeled_df.copy()

clean_df = clean_df.drop_duplicates(subset=["url"])
clean_df = clean_df.drop_duplicates(subset=["id"])
clean_df = clean_df[(clean_df["title"].str.len() > 0) & (clean_df["text"].str.len() >= 200)]

print("Shape before cleaning:", labeled_df.shape)
print("Shape after cleaning :", clean_df.shape)

clean_df["label"].value_counts().sort_index()

Shape before cleaning: (140820, 10)
Shape after cleaning : (130085, 10)


label
0    118306
1     11779
Name: count, dtype: int64

## Thống kê theo yêu cầu báo cáo

In [31]:
label_counts = clean_df["label"].value_counts().sort_index()
label_ratios = clean_df["label"].value_counts(normalize=True).sort_index().round(4)

summary_df = pd.DataFrame({
    "label": label_counts.index,
    "count": label_counts.values,
    "ratio": label_ratios.values
})

print("Tổng số mẫu sạch đã gán nhãn:", len(clean_df))
summary_df

Tổng số mẫu sạch đã gán nhãn: 130085


,label,count,ratio
0,0,118306,0.9095
1,1,11779,0.0905


## Lưu file
- final_df là bản đã gán nhãn và chuẩn hóa cột, nhưng chưa làm sạch triệt để.
- clean_df là bản đã qua bước cleaning cơ bản để dùng cho huấn luyện mô hình, ví dụ bỏ trùng id/url, bỏ bài rỗng, bỏ bài quá ngắn.

In [32]:
from pathlib import Path

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

final_df.to_csv(OUTPUT_DIR / "news_labeled_all.csv", index=False, encoding="utf-8-sig")
clean_df.to_csv(OUTPUT_DIR / "news_labeled_clean.csv", index=False, encoding="utf-8-sig")

## 